# 모델별 운영 결과 비교

## 분석 목적
동일 Case에서 model profile별 실제 runtime/final_action 결과 분포를 비교한다.

## 가설
H0: 동일 Case의 모델별 운영 완료 확률이 같다. 모델 품질/Utility 가설은 관측 변수가 없다.

## 사용할 변수
eval_case_id, model_profile_id, runtime_status, final_action, provider_status, error_category

## 통계기법 선택 이유
독립 Case임을 확인한 2모델 binary 비교는 exact McNemar와 paired rate difference. 3모델 이상은 충분한 informative Case에서 Cochran Q. 반복 Case 구조이므로 독립 표본 Chi-square를 기본 적용하지 않는다. 범주 분포는 기술통계로 제시한다.

## 해석 기준
검정 가정 미확인은 p-value 보류. 수행한 검정은 Holm 보정과 효과크기로 해석한다. workload_id와 quality score가 없어 Workload/품질 평가는 불가. 합성 결과는 실증 근거가 아니다.

기본 입력은 **합성 Consumer 시험 fixture**이다. 실제 Bundle은 `ADP_AI_BUNDLE_SOURCE`로 지정한다. Case 독립성/대칭성은 자동 추정하지 않는다.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "02_ai/src").is_dir())
sys.path.insert(0, str(ROOT / "02_ai/src"))
from adp_da.bundle_analysis import analyze_bundle  # noqa: E402
from adp_da.bundle_dataset import execution_dataframe  # noqa: E402
from adp_da.bundle_loader import load_bundle  # noqa: E402

source = os.environ.get("ADP_AI_BUNDLE_SOURCE")
synthetic = source is None or os.environ.get("ADP_AI_SYNTHETIC") == "1"
if source is None:
    source = str(ROOT / "02_ai/tests/fixtures/evaluation_bundle.synthetic.json")
bundle, metadata = load_bundle(
    source, ROOT / "02_ai/data/interim/ai_evaluation/raw",
    evaluation_run_id=os.environ.get("ADP_AI_EVALUATION_RUN_ID"),
    token=os.environ.get("ADP_BE_TOKEN"),
    local_admin_user_id=os.environ.get("ADP_BE_LOCAL_ADMIN_USER_ID"),
    local_admin_roles=os.environ.get("ADP_BE_LOCAL_ADMIN_ROLES"),
)
frame = execution_dataframe(bundle)
artifacts = analyze_bundle(
    frame, synthetic=synthetic,
    independent_cases=os.environ.get("ADP_AI_INDEPENDENT_CASES") == "1",
    symmetric_differences=os.environ.get("ADP_AI_SYMMETRIC_DIFFERENCES") == "1",
)
print("SYNTHETIC FIXTURE — SOFTWARE VALIDATION ONLY" if synthetic else "USER-SUPPLIED BUNDLE")
display({k: bundle["manifest"][k] for k in ("bundle_id", "content_digest", "execution_count")})


In [ ]:
display(artifacts["model_comparison"])
display(frame.pivot(index="eval_case_id", columns="model_profile_id", values="runtime_status"))
